In [1]:
"""
Statistical Significance for Brain Signal Classification
=========================================================
Python implementation of the methods described in:

Combrisson & Jerbi (2015) - "Exceeding chance level by chance: The caveat of
theoretical chance levels in brain signal classification and statistical
assessment of decoding accuracy"
Journal of Neuroscience Methods, 250, 126-136.

Two approaches are implemented:
  1. Binomial cumulative distribution (analytical)
  2. Permutation tests (empirical/data-driven)

Plus simulation utilities to reproduce the paper's core findings.
"""

import numpy as np
from scipy.stats import binom
from scipy.stats import norm as sp_norm
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, LeaveOneOut, cross_val_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")


# =============================================================================
# 1. BINOMIAL-BASED SIGNIFICANCE THRESHOLD
# =============================================================================
def age(ind):
    if '3m' in ind:
        return 'young'
    else:
        return 'old'

def condition(ind):
    if '5xFAD' in ind:
        return '5xFAD'
    else:
        return 'Wild-Type'


def binomial_threshold(n_samples: int, n_classes: int, alpha: float = 0.05) -> float:
    """
    Compute the minimum decoding accuracy (%) required to be statistically
    significant, using the binomial cumulative distribution.

    This accounts for finite sample size — unlike the naive theoretical
    chance level of 100/n_classes %.

    Parameters
    ----------
    n_samples : int
        Total number of test samples.
    n_classes : int
        Number of classes (e.g. 2 for binary classification).
    alpha : float
        Significance level (e.g. 0.05, 0.01, 0.001).

    Returns
    -------
    float
        Minimum accuracy (%) that is statistically significant at level alpha.

    Example
    -------
    >>> binomial_threshold(n_samples=40, n_classes=2, alpha=0.001)
    75.0
    # Matches the paper's Table 1: for n=40, 2-class, p<0.001 → 75%
    """
    chance = 1.0 / n_classes
    # binom.ppf gives the value k such that P(X <= k) >= 1 - alpha
    k = binom.ppf(1 - alpha, n_samples, chance)
    threshold = (k / n_samples) * 100.0
    return round(threshold, 1)


def significance_table(
    sample_sizes: list = None,
    n_classes_list: list = None,
    alphas: list = None,
) -> dict:
    """
    Reproduce Table 1 from the paper: a lookup table of significance thresholds.

    Parameters
    ----------
    sample_sizes : list of int
        Sample sizes to evaluate. Defaults to paper values.
    n_classes_list : list of int
        Numbers of classes. Defaults to [2, 4, 8].
    alphas : list of float
        Significance levels. Defaults to [0.05, 0.01, 0.001, 0.0001].

    Returns
    -------
    dict
        Nested dict: result[n][c][alpha] = threshold (%).
    """
    table = {}
    for n in sample_sizes:
        table[n] = {}
        for c in n_classes_list:
            table[n][c] = {}
            for a in alphas:
                table[n][c][a] = binomial_threshold(n, c, a)
    return table


def print_significance_table(table: dict = None) -> None:
    """Pretty-print the significance threshold table (replicates Table 1)."""
    if table is None:
        table = significance_table()

    sample_sizes = sorted(table.keys())
    alphas = [0.05, 0.01, 0.001, 0.0001]
    alpha_labels = ["p<0.05", "p<0.01", "p<0.001", "p<0.0001"]

    for c in [2, 4, 8]:
        print(f"\n{'='*60}")
        print(f"  {c}-CLASS CLASSIFICATION — Significance Thresholds (%)")
        print(f"{'='*60}")
        header = f"{'n':>6}  " + "  ".join(f"{l:>9}" for l in alpha_labels)
        print(header)
        print("-" * len(header))
        for n in sample_sizes:
            row = f"{n:>6}  " + "  ".join(
                f"{table[n][c][a]:>8.1f}%" for a in alphas
            )
            print(row)


# =============================================================================
# 2. PERMUTATION TEST
# =============================================================================

def permutation_test(
    X: np.ndarray,
    y: np.ndarray,
    classifier=None,
    n_permutations: int = 1000,
    cv_folds: int = 10,
    alpha: float = 0.05,
    random_state: int = 42,
) -> dict:
    """
    Assess classification significance via label permutation testing.

    Shuffles class labels repeatedly and computes the null distribution of
    classification accuracies. The original accuracy is then compared to
    this distribution to derive an empirical p-value.

    Parameters
    ----------
    X : np.ndarray, shape (n_samples, n_features)
        Feature matrix.
    y : np.ndarray, shape (n_samples,)
        Class labels.
    classifier : sklearn estimator, optional
        Defaults to LinearDiscriminantAnalysis.
    n_permutations : int
        Number of label permutations (paper used 10,000; 1,000 is faster).
    cv_folds : int
        Number of cross-validation folds.
    alpha : float
        Significance level for the threshold.
    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    dict with keys:
        - 'observed_accuracy': float — accuracy on original labels (%)
        - 'null_distribution': np.ndarray — accuracies under null (%)
        - 'p_value': float — empirical p-value
        - 'threshold': float — significance boundary at given alpha (%)
        - 'is_significant': bool

    """
    rng = np.random.default_rng(random_state)

    if classifier is None:
        classifier = LinearDiscriminantAnalysis()

    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)

    # Observed accuracy
    observed = np.mean(cross_val_score(classifier, X, y, cv=cv, scoring="accuracy"))

    # Null distribution via label permutations
    null_accuracies = np.zeros(n_permutations)
    for i in range(n_permutations):
        y_perm = rng.permutation(y)
        null_accuracies[i] = np.mean(
            cross_val_score(classifier, X, y_perm, cv=cv, scoring="accuracy")
        )

    # Empirical p-value: fraction of permutations >= observed accuracy
    p_value = np.mean(null_accuracies >= observed)

    # Significance threshold at given alpha
    threshold = np.percentile(null_accuracies, (1 - alpha) * 100)

    return {
        "observed_accuracy": observed * 100,
        "null_distribution": null_accuracies * 100,
        "p_value": p_value,
        "threshold": threshold * 100,
        "is_significant": observed > threshold,
    }



## 1D sinusoidal

In [2]:
import json
import joblib
import numpy as np
import pandas as pd
# ── config ────────────────────────────────────────────────────────────────────

dataset_map = {
    'energy':   'Energy_level3.json',
    'coactiv':  'Coactiv.json',
    'entropy':  'Entropy.json',
    'kl_div':   'kl_div.json',
}

label_map = {
    'age':  age,        # your existing function
    'cond': condition,  # your existing function
}

# experiments you actually have models for
experiments = {
    'energy':   [i for i in range(21)],
    'coactiv age':  [i for i in range(21)],
    'coactiv cond':  [i for i in range(21)],
    'entropy age':  [i for i in range(21)],
    'entropy cond':  [i for i in range(21)],
    'kl_div':   [i for i in range(21)],
}

models_dir = '1D_ind_sin_analyses/models/'

# ── load datasets once ────────────────────────────────────────────────────────

dataframes = {
    name: pd.read_json('1D_ind_sin_analyses/'+path)
    for name, path in dataset_map.items()
}

# ── permutation test loop ─────────────────────────────────────────────────────

results = {}

for dataset_name, df in dataframes.items():
    print(dataset_name,"================")
    for label_name, label_fn in label_map.items():
        if (dataset_name == "entropy" or dataset_name == "coactiv") and label_name == "age":
            experiments_name = dataset_name+' age'
        elif (dataset_name == "entropy" or dataset_name == "coactiv") and label_name == "cond":
            experiments_name = dataset_name+' cond'
        else:
            experiments_name = dataset_name
        for exp in experiments[experiments_name]:

            model_path = f'{models_dir}{dataset_name}_{label_name}_svm_{exp}.pkl'

            # skip if model doesn't exist for this combination
            try:
                svm = joblib.load(model_path)
            except FileNotFoundError:
                continue

            # features: column exp as 2D array (n_samples, 1)
            if isinstance(df.columns[1],str):
                X = df[str(exp)].values.reshape(-1, 1)
            else:
                X = df[exp].values.reshape(-1, 1)
            # labels: binary array from type column
            if "type" not in df.columns:
                df["type"] = df.index

            if label_name =="age":
                y = df['type'].apply(age)
            else:
                y = df['type'].apply(condition)

            result = permutation_test(
                X              = X,
                y              = y,
                classifier     = svm,
                n_permutations = 1000,
                cv_folds       = 10,
                alpha          = 0.05,
            )

            key = f'{dataset_name}_{label_name}_{exp}'
            results[key] = {
                'dataset':    dataset_name,
                'label':      label_name,
                'experiment': exp,
                'p_value':    result['p_value'],
                'observed':   result['observed_accuracy'],
                'threshold':  result['threshold'],
                'significant': result['is_significant'],
            }

            print(
                f"{key:<35} "
                f"acc={result['observed_accuracy']:.1f}%  "
                f"p={result['p_value']:.4f}  "
                f"sig={result['is_significant']}"
            )

# ── save results ──────────────────────────────────────────────────────────────

results_df = pd.DataFrame(results).T
results_df.to_csv('permutation_test_results.csv', index=True)
print("\nSaved: permutation_test_results.csv")

energy ================
energy_age_0                        acc=48.3%  p=0.5540  sig=False
energy_age_1                        acc=55.0%  p=0.3370  sig=False
energy_age_2                        acc=73.3%  p=0.0560  sig=False
energy_age_3                        acc=68.3%  p=0.0820  sig=False
energy_age_4                        acc=58.3%  p=0.2310  sig=False
energy_age_5                        acc=71.7%  p=0.0400  sig=True
energy_age_6                        acc=61.7%  p=0.1650  sig=False
energy_age_7                        acc=68.3%  p=0.0730  sig=False
energy_age_8                        acc=60.0%  p=0.1830  sig=False
energy_age_9                        acc=78.3%  p=0.0050  sig=True
energy_age_10                       acc=63.3%  p=0.1910  sig=False
energy_age_11                       acc=58.3%  p=0.1840  sig=False
energy_age_12                       acc=71.7%  p=0.0580  sig=False
energy_age_13                       acc=63.3%  p=0.1710  sig=False
energy_age_14                       acc=

In [3]:
results_df = pd.read_csv("permutation_test_results.csv")
results_df

,Unnamed: 0,dataset,label,experiment,p_value,observed,threshold,significant
0,energy_age_0,energy,age,0,0.554,48.333333,70.000000,False
1,energy_age_1,energy,age,1,0.337,55.000000,66.666667,False
2,energy_age_2,energy,age,2,0.056,73.333333,73.333333,False
3,energy_age_3,energy,age,3,0.082,68.333333,71.666667,False
4,energy_age_4,energy,age,4,0.231,58.333333,68.333333,False
...,...,...,...,...,...,...,...,...
121,entropy_cond_16,entropy,cond,16,0.436,53.333333,63.333333,False
122,entropy_cond_17,entropy,cond,17,0.339,53.333333,66.666667,False
123,entropy_cond_18,entropy,cond,18,0.311,58.333333,71.666667,False
124,entropy_cond_19,entropy,cond,19,0.996,41.666667,58.333333,False


In [4]:
from statsmodels.stats.multitest import multipletests

for dataset_name in results_df['dataset'].unique():
    mask = results_df['dataset'] == dataset_name
    p_vals = results_df.loc[mask, 'p_value'].values

    rejected, p_corrected, _, _ = multipletests(p_vals, alpha=0.05, method='fdr_bh')

    results_df.loc[mask, 'p_corrected'] = p_corrected
    results_df.loc[mask, 'sig_fdr']     = rejected

print(results_df[['dataset', 'label', 'experiment',
                   'observed', 'p_value', 'p_corrected', 'sig_fdr']])

results_df.to_csv('permutation_test_results_fdr.csv', index=True)

     dataset label  experiment   observed  p_value  p_corrected sig_fdr
0     energy   age           0  48.333333    0.554     0.775600   False
1     energy   age           1  55.000000    0.337     0.524222   False
2     energy   age           2  73.333333    0.056     0.246000   False
3     energy   age           3  68.333333    0.082     0.246000   False
4     energy   age           4  58.333333    0.231     0.404250   False
..       ...   ...         ...        ...      ...          ...     ...
121  entropy  cond          16  53.333333    0.436     0.654000   False
122  entropy  cond          17  53.333333    0.339     0.569520   False
123  entropy  cond          18  58.333333    0.311     0.558250   False
124  entropy  cond          19  41.666667    0.996     1.000000   False
125  entropy  cond          20  65.000000    0.098     0.218842   False

[126 rows x 7 columns]


In [7]:
results_df[results_df['sig_fdr']==True]

,Unnamed: 0,dataset,label,experiment,p_value,observed,threshold,significant,p_corrected,sig_fdr
47,coactiv_age_5,coactiv,age,5,0.001,86.666667,73.333333,True,0.042,True
54,coactiv_age_12,coactiv,age,12,0.002,83.333333,71.666667,True,0.042,True


## 1D brownian

In [5]:
import json
import joblib
import numpy as np
import pandas as pd
# ── config ────────────────────────────────────────────────────────────────────

dataset_map = {
    'energy':   'Energy_level3.json',
    'Coactiv':  'Coactiv.json',
    'entropy':  'Entropy.json',
    'kldiv':   'kldiv.json',
    'H': 'Hurst_regression.json'
}

label_map = {
    'age':  age,        # your existing function
    'cond': condition,  # your existing function
}

# experiments you actually have models for
experiments = {
    'energy':   [i for i in range(6)],
    'Coactiv':  [i for i in range(6)],
    'entropy':  [i for i in range(6)],
    'kldiv':   [i for i in range(6)],
    'H':   [i for i in range(6)],
}

models_dir = '1D_brownian_analyses/models/'

# ── load datasets once ────────────────────────────────────────────────────────

dataframes = {
    name: pd.read_json('1D_brownian_analyses/'+path)
    for name, path in dataset_map.items()
}

# ── permutation test loop ─────────────────────────────────────────────────────

results = {}

for dataset_name, df in dataframes.items():
    print(dataset_name,"================")
    for label_name, label_fn in label_map.items():
        experiments_name = dataset_name
        for exp in experiments[experiments_name]:
            
            model_path = f'{models_dir}{dataset_name}_{label_name}_svm_{exp}.pkl'

            # skip if model doesn't exist for this combination
            try:
                svm = joblib.load(model_path)
            except FileNotFoundError:
                continue

            # features: column exp as 2D array (n_samples, 1)
            if isinstance(df.columns[1],str):
                X = df[str(exp)].values.reshape(-1, 1)
            else:
                X = df[exp].values.reshape(-1, 1)
            # labels: binary array from type column
            if "type" not in df.columns:
                df["type"] = df.index

            if label_name =="age":
                y = df['type'].apply(age)
            else:
                y = df['type'].apply(condition)

            result = permutation_test(
                X              = X,
                y              = y,
                classifier     = svm,
                n_permutations = 1000,
                cv_folds       = 10,
                alpha          = 0.05,
            )

            key = f'{dataset_name}_{label_name}_{exp}'
            results[key] = {
                'dataset':    dataset_name,
                'label':      label_name,
                'experiment': exp,
                'p_value':    result['p_value'],
                'observed':   result['observed_accuracy'],
                'threshold':  result['threshold'],
                'significant': result['is_significant'],
            }

            print(
                f"{key:<35} "
                f"acc={result['observed_accuracy']:.1f}%  "
                f"p={result['p_value']:.4f}  "
                f"sig={result['is_significant']}"
            )

# ── save results ──────────────────────────────────────────────────────────────

results_df = pd.DataFrame(results).T
results_df.to_csv('permutation_test_results1dbrown.csv', index=True)
print("\nSaved: permutation_test_results.csv")

energy ================
energy_age_0                        acc=41.7%  p=0.8760  sig=False
energy_age_1                        acc=61.7%  p=0.2040  sig=False
energy_age_2                        acc=65.0%  p=0.1390  sig=False
energy_age_3                        acc=70.0%  p=0.0560  sig=False
energy_age_4                        acc=70.0%  p=0.0530  sig=False
energy_age_5                        acc=66.7%  p=0.1070  sig=False
energy_cond_0                       acc=73.3%  p=0.0100  sig=True
energy_cond_1                       acc=66.7%  p=0.0910  sig=False
energy_cond_2                       acc=58.3%  p=0.2640  sig=False
energy_cond_3                       acc=48.3%  p=0.4880  sig=False
energy_cond_4                       acc=51.7%  p=0.4420  sig=False
energy_cond_5                       acc=58.3%  p=0.0160  sig=True
Coactiv ================
Coactiv_age_0                       acc=50.0%  p=0.4880  sig=False
Coactiv_age_1                       acc=61.7%  p=0.1720  sig=False
Coactiv_age_2  

In [6]:
results_df = pd.read_csv("permutation_test_results1dbrown.csv")
results_df

,Unnamed: 0,dataset,label,experiment,p_value,observed,threshold,significant
0,energy_age_0,energy,age,0,0.876,41.666667,55.000000,False
1,energy_age_1,energy,age,1,0.204,61.666667,70.000000,False
2,energy_age_2,energy,age,2,0.139,65.000000,70.000000,False
3,energy_age_3,energy,age,3,0.056,70.000000,70.000000,False
4,energy_age_4,energy,age,4,0.053,70.000000,70.000000,False
5,energy_age_5,energy,age,5,0.107,66.666667,70.000000,False
6,energy_cond_0,energy,cond,0,0.010,73.333333,65.000000,True
7,energy_cond_1,energy,cond,1,0.091,66.666667,68.416667,False
8,energy_cond_2,energy,cond,2,0.264,58.333333,71.666667,False
9,energy_cond_3,energy,cond,3,0.488,48.333333,55.000000,False


In [7]:
from statsmodels.stats.multitest import multipletests

for dataset_name in results_df['dataset'].unique():
    mask = results_df['dataset'] == dataset_name
    p_vals = results_df.loc[mask, 'p_value'].values

    rejected, p_corrected, _, _ = multipletests(p_vals, alpha=0.05, method='fdr_bh')

    results_df.loc[mask, 'p_corrected'] = p_corrected
    results_df.loc[mask, 'sig_fdr']     = rejected

print(results_df[['dataset', 'label', 'experiment',
                   'observed', 'p_value', 'p_corrected', 'sig_fdr']])

results_df.to_csv('permutation_test_results_fdr1dbrownian.csv', index=True)

    dataset label  experiment   observed  p_value  p_corrected sig_fdr
0    energy   age           0  41.666667    0.876     0.876000   False
1    energy   age           1  61.666667    0.204     0.306000   False
2    energy   age           2  65.000000    0.139     0.238286   False
3    energy   age           3  70.000000    0.056     0.168000   False
4    energy   age           4  70.000000    0.053     0.168000   False
5    energy   age           5  66.666667    0.107     0.214000   False
6    energy  cond           0  73.333333    0.010     0.096000   False
7    energy  cond           1  66.666667    0.091     0.214000   False
8    energy  cond           2  58.333333    0.264     0.352000   False
9    energy  cond           3  48.333333    0.488     0.532364   False
10   energy  cond           4  51.666667    0.442     0.530400   False
11   energy  cond           5  58.333333    0.016     0.096000   False
12  Coactiv   age           0  50.000000    0.488     0.532364   False
13  Co

In [8]:
results_df[results_df['sig_fdr']==True]

,Unnamed: 0,dataset,label,experiment,p_value,observed,threshold,significant,p_corrected,sig_fdr
18,Coactiv_cond_0,Coactiv,cond,0,0.001,83.333333,70.0,True,0.012,True


## 2D brownian

In [33]:
import json
import joblib
import numpy as np
import pandas as pd
# ── config ────────────────────────────────────────────────────────────────────

dataset_map = {
    'energy':   'Energy_level3.json',
    'Coactiv':  'Coactiv.json',
    'entropy':  'Entropy.json',
    'kldiv':   'kldiv.json',
    'H': 'h_est.json'
}

label_map = {
    'age':  age,        # your existing function
    'cond': condition,  # your existing function
}

# experiments you actually have models for
experiments = {
    'energy':   [i for i in range(6)],
    'Coactiv':  [i for i in range(6)],
    'entropy':  [i for i in range(6)],
    'kldiv':   [i for i in range(6)],
    'H':   [i for i in range(6)],
}

models_dir = '2D_analyses/models/'

# ── load datasets once ────────────────────────────────────────────────────────

dataframes = {
    name: pd.read_json('2D_analyses/'+path)
    for name, path in dataset_map.items()
}

# ── permutation test loop ─────────────────────────────────────────────────────

import os
import joblib
import pandas as pd

# ── índice de modelos disponibles ──────────────────────────────────────────
import os
import joblib
import pandas as pd

# ── índice de modelos disponibles ──────────────────────────────────────────
def normalize(s):
    return s.lower().replace('_', '')

model_files = [f for f in os.listdir(models_dir) if f.endswith(('.joblib', '.pkl'))]
model_lookup = {normalize(os.path.splitext(f)[0]): f for f in model_files}

def find_model_key(dataset_name, label_name, model_lookup):
    dataset_norm = normalize(dataset_name)
    label_norm = normalize(label_name)
    for key in model_lookup:
        if key.startswith(dataset_norm) and key[len(dataset_norm):].startswith(label_norm):
            return key
    return None

results = {}

for dataset_name, df in dataframes.items():
    print(dataset_name, "================")
    for label_name, label_fn in label_map.items():
        experiments_name = dataset_name

        # ── modelo: se busca por dataset + label únicamente ──
        key_norm = find_model_key(dataset_name, label_name, model_lookup)
        if key_norm is None:
            continue

        model_path = os.path.join(models_dir, model_lookup[key_norm])
        loaded = joblib.load(model_path)

        # el .joblib guarda un dict con metadata; el estimador está en 'model'
        svm = loaded['model'] if isinstance(loaded, dict) else loaded

        for exp in experiments[experiments_name]:

            # features: columna exp como array 2D (n_samples, 1)
            if isinstance(df.columns[1], str):
                X = df[df.columns[1]].values.reshape(-1, 1)
            else:
                X = df[df.columns[1]].values.reshape(-1, 1)

            # labels: array binario desde la columna type
            if "type" not in df.columns:
                df["type"] = df.index

            if label_name == "age":
                y = df['type'].apply(age)
            else:
                y = df['type'].apply(condition)

            result = permutation_test(
                X              = X,
                y              = y,
                classifier     = svm,
                n_permutations = 1000,
                cv_folds       = 10,
                alpha          = 0.05,
            )

            key = f'{dataset_name}_{label_name}_{exp}'
            results[key] = {
                'dataset':    dataset_name,
                'label':      label_name,
                'experiment': exp,
                'p_value':    result['p_value'],
                'observed':   result['observed_accuracy'],
                'threshold':  result['threshold'],
                'significant': result['is_significant'],
            }

            print(
                f"{key:<35} "
                f"acc={result['observed_accuracy']:.1f}%  "
                f"p={result['p_value']:.4f}  "
                f"sig={result['is_significant']}"
            )

# ── guardar resultados ──────────────────────────────────────────────────────
results_df = pd.DataFrame(results).T
results_df.to_csv('permutation_test_results2D.csv', index=True)
print("\nSaved: permutation_test_results2D.csv")

energy ================
Coactiv ================
Coactiv_age_0                       acc=26.7%  p=0.9560  sig=False
Coactiv_age_1                       acc=26.7%  p=0.9560  sig=False
Coactiv_age_2                       acc=26.7%  p=0.9560  sig=False
Coactiv_age_3                       acc=26.7%  p=0.9560  sig=False
Coactiv_age_4                       acc=26.7%  p=0.9560  sig=False
Coactiv_age_5                       acc=26.7%  p=0.9560  sig=False
Coactiv_cond_0                      acc=53.3%  p=0.3010  sig=False
Coactiv_cond_1                      acc=53.3%  p=0.3010  sig=False
Coactiv_cond_2                      acc=53.3%  p=0.3010  sig=False
Coactiv_cond_3                      acc=53.3%  p=0.3010  sig=False
Coactiv_cond_4                      acc=53.3%  p=0.3010  sig=False
Coactiv_cond_5                      acc=53.3%  p=0.3010  sig=False
entropy ================
entropy_age_0                       acc=26.7%  p=0.9690  sig=False
entropy_age_1                       acc=26.7%  p=0.9690

KeyboardInterrupt: 

In [29]:
model_lookup

{'coactivage': 'coactiv__age.joblib',
 'coactivcondition': 'coactiv__condition.joblib',
 'energylevel3age': 'energy_level3__age.joblib',
 'energylevel3condition': 'energy_level3__condition.joblib',
 'entropyage': 'Entropy__age.joblib',
 'entropycondition': 'Entropy__condition.joblib',
 'hestxage': 'h_est_x__age.joblib',
 'hestxcondition': 'h_est_x__condition.joblib',
 'hestyage': 'h_est_y__age.joblib',
 'hestycondition': 'h_est_y__condition.joblib',
 'kldivage': 'kldiv__age.joblib',
 'kldivcondition': 'kldiv__condition.joblib'}

In [30]:
model_lookup[key_norm]

'coactiv__age.joblib'